# Gate 3.1 — Evaluation Alignment Verification

**Purpose:** Check that September keyword and TF-IDF baselines and the selected October ModernBERT Candidate A use the approved validation evaluation units.

**Rules:** Do not retrain models, retune thresholds, or overwrite frozen September artifacts. Run this notebook from the cloned repository's `notebooks/` folder. This notebook reads existing artifacts and performs checks; it does not commit or push anything.

**Status:** September checks can be reproduced now. October prediction checks require the missing full-validation prediction CSV or the original fine-tuned checkpoint.

## 1. Imports and repository paths

In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

assert (ROOT / "data/gate3").is_dir(), f"Repository root not found: {ROOT}"
print("Repository:", ROOT)

## 2. Approved validation chunks and labels

The approved September chunking configuration uses 768-token chunks, 512-token overlap, and a 256-token step. A validation unit is the pair `(chunk_id, category)`.

In [ ]:
val_chunks = pd.read_csv(ROOT / "data/gate3/validation_chunks_768_512.csv")
val_labels = pd.read_csv(ROOT / "data/gate3/validation_chunk_labels_768_512.csv")
KEYS = ["chunk_id", "category"]

assert val_chunks["chunk_id"].is_unique
assert not val_labels.duplicated(KEYS).any()
assert len(val_chunks) == 2481, f"Unexpected chunks: {len(val_chunks)}"
assert len(val_labels) == 24810, f"Unexpected evaluation pairs: {len(val_labels)}"
assert val_labels.groupby("chunk_id")["category"].nunique().eq(10).all()
assert set(val_labels["chunk_id"]) == set(val_chunks["chunk_id"])

for col, expected in [("chunk_size", 768), ("overlap", 512), ("step_size", 256)]:
    assert val_chunks[col].eq(expected).all(), f"Unexpected {col}"
assert val_chunks["config_version"].eq("gate3_chunking_v1").all()

print("PASS: Approved validation units loaded")
print("Chunks:", len(val_chunks))
print("Pairs:", len(val_labels))
print("Unmasked:", int(val_labels["mask"].eq(0).sum()))
print("Masked:", int(val_labels["mask"].eq(1).sum()))

## 3. September keyword vs. TF-IDF alignment

This check reads the **committed** combined validation rows; it does not rescore or retrain either baseline. The separate `gate3_evaluation.ipynb` reproduced keyword scores exactly and TF-IDF unmasked margins within tolerance in the earlier session.

In [ ]:
rows = pd.read_csv(ROOT / "reports/gate3_evaluation/validation_rows.csv")
keyword_rows = rows.loc[rows["baseline"] == "keyword"].copy()
tfidf_rows = rows.loc[rows["baseline"] == "tfidf"].copy()

approved_keys = set(map(tuple, val_labels[KEYS].itertuples(index=False, name=None)))
keyword_keys = set(map(tuple, keyword_rows[KEYS].itertuples(index=False, name=None)))
tfidf_keys = set(map(tuple, tfidf_rows[KEYS].itertuples(index=False, name=None)))

assert not keyword_rows.duplicated(KEYS).any()
assert not tfidf_rows.duplicated(KEYS).any()
assert keyword_keys == tfidf_keys == approved_keys

comparison = keyword_rows.merge(tfidf_rows, on=KEYS, suffixes=("_keyword", "_tfidf"), validate="one_to_one")
for field in ["target", "mask", "title", "context_group_id"]:
    a, b = comparison[f"{field}_keyword"], comparison[f"{field}_tfidf"]
    assert a.equals(b), f"September {field} mismatch"

# Independently verify that both baselines' joined labels agree with approved labels.
for name, baseline in [("keyword", keyword_rows), ("tfidf", tfidf_rows)]:
    check = baseline.merge(val_labels[KEYS + ["target", "mask"]], on=KEYS, suffixes=("_baseline", "_approved"), validate="one_to_one")
    assert len(check) == len(val_labels)
    for field in ["target", "mask"]:
        assert check[f"{field}_baseline"].equals(check[f"{field}_approved"]), f"{name}: {field} mismatch"

print("PASS: September baselines have identical approved evaluation pairs, targets, masks, titles, and context groups")
print("Evaluation pairs:", len(approved_keys))

## 4. Frozen October category names and order

The transformer category order differs from the September alphabetical order. Always align predictions by category **name**, not by column position.

In [ ]:
candidate_dir = ROOT / "configs/Accenture_October_Transformer/selected_transformer_candidate_a"
with (candidate_dir / "category_order.json").open(encoding="utf-8") as f:
    category_data = json.load(f)

# The committed category order file is expected to contain a list.
if isinstance(category_data, list):
    frozen_category_order = category_data
elif isinstance(category_data, dict) and isinstance(category_data.get("category_order"), list):
    frozen_category_order = category_data["category_order"]
else:
    raise ValueError("Unexpected category_order.json format; inspect it before proceeding")

approved_categories = set(val_labels["category"].unique())
assert len(frozen_category_order) == 10
assert len(set(frozen_category_order)) == 10
assert set(frozen_category_order) == approved_categories
print("PASS: Frozen transformer category names match approved categories")
print("Frozen order:", frozen_category_order)

## 5. October full-validation predictions (conditional)

**Blocked until available:** The original fine-tuned Candidate A weights are not in the local checkout, and the previous full-validation predictions were not exported. Ask the teammate who trained Candidate A for the saved checkpoint or the full-validation prediction CSV. Do **not** use generic pretrained ModernBERT weights as a substitute.

When the CSV arrives, place it at `reports/gate3_alignment/transformer_validation_predictions.csv` and rerun this cell.

In [ ]:
prediction_path = ROOT / "reports/gate3_alignment/transformer_validation_predictions.csv"
transformer_predictions = None

if not prediction_path.is_file():
    print("PENDING: Full-validation transformer predictions CSV is unavailable")
else:
    transformer_predictions = pd.read_csv(prediction_path)
    assert set(KEYS).issubset(transformer_predictions.columns)
    assert len(transformer_predictions) == len(val_labels)
    assert not transformer_predictions.duplicated(KEYS).any()
    transformer_keys = set(map(tuple, transformer_predictions[KEYS].itertuples(index=False, name=None)))
    assert transformer_keys == approved_keys
    assert set(transformer_predictions["category"]) == approved_categories
    print("PASS: October transformer has exactly the approved evaluation pairs")
    print("Prediction rows:", len(transformer_predictions))

## 6. Source mapping, masks, and input-text provenance

The approved chunks contain document title, token and character offsets, text, and text hashes. A successful join proves that each prediction has an approved source record; it does **not**, by itself, prove what text was actually fed into a separately executed model. That input-text check passed in the earlier live October notebook session and needs independent reproduction once predictions and the frozen checkpoint are available.

In [ ]:
source_cols = ["chunk_id", "title", "token_start", "token_end", "char_start", "char_end", "text_hash"]
assert all(c in val_chunks.columns for c in source_cols)
assert val_chunks["chunk_id"].is_unique

if transformer_predictions is None:
    print("PENDING: Transformer prediction-to-source and mask joins (prediction CSV missing)")
else:
    aligned = transformer_predictions.merge(
        val_labels[KEYS + ["target", "mask"]],
        on=KEYS, how="left", validate="one_to_one", indicator=True
    )
    assert aligned["_merge"].eq("both").all()
    assert aligned["target"].notna().all() and aligned["mask"].notna().all()
    aligned = aligned.drop(columns="_merge").merge(
        val_chunks[source_cols], on="chunk_id", how="left", validate="many_to_one", indicator=True
    )
    assert aligned["_merge"].eq("both").all()
    assert len(aligned) == 24810
    print("PASS: Transformer predictions map to approved targets, masks, and source records")
    print("Unmasked:", int(aligned["mask"].eq(0).sum()))
    print("Masked:", int(aligned["mask"].eq(1).sum()))

## 7. SHA-256 artifact inventory (read-only)

This cell prints hashes of available files for review. The standalone manifest in `reports/gate3_alignment/artifact_manifest.json` is a separate deliverable; do not silently overwrite it. A source commit hash and independently verifiable frozen checkpoint are still needed for complete reproducibility.

In [ ]:
artifact_names = [
    "data/gate3/validation_chunks_768_512.csv",
    "data/gate3/validation_chunk_labels_768_512.csv",
    "reports/gate3_evaluation/validation_rows.csv",
    "configs/keyword_rules_v1.json",
    "configs/keyword_thresholds_v1.json",
    "reports/tf_idf_baseline/selected_model.npz",
    "reports/tf_idf_baseline/selected_model.json",
    "reports/gate3_alignment/transformer_validation_predictions.csv",
]

for name in artifact_names:
    path = ROOT / name
    if not path.is_file():
        print("MISSING:", name)
    else:
        h = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        print("SHA256:", h.hexdigest(), name)

## 8. Results and limitations

| Check | Status |
|---|---|
| September keyword scores reproduced in prior `gate3_evaluation.ipynb` run | **PASS** — 24,810 exact matches |
| September TF-IDF margins reproduced in prior run | **PASS** — 24,646 unmasked margins within tolerance; identical detections |
| September `(chunk_id, category)`, targets, masks, titles, context groups | **PASS** — independently checked above |
| Frozen October category names | **PASS** — checked above |
| October 24,810-pair coverage and key equality | **PASS in earlier session; pending independent reproduction** |
| October labels, masks, source mapping, approved input text | **PASS in earlier session; pending independent reproduction** |
| Original fine-tuned Candidate A weights | **PENDING** — teammate to provide |
| Full-validation transformer prediction CSV | **PENDING** — teammate to provide or regenerate from exact frozen weights |
| Complete artifact manifest and independent review | **PENDING** |

### Conclusion

The earlier validation checks support alignment of the October transformer with the September keyword and TF-IDF baselines on the approved evaluation units. September results were preserved without retraining or threshold retuning. Full independent transformer reproduction is **not yet complete** because the original Candidate A weights and full-validation predictions are unavailable locally.

**Next action:** Obtain the exact frozen Candidate A checkpoint or the full-validation predictions from the teammate, verify artifact hashes, rerun the conditional checks, and update the alignment report and manifest. Do not merge as a fully reproduced evaluation until these outstanding items are addressed.